# 03 - Feature Engineering

**Kenya Food Price Inflation Tracker**

## Objective
Create features for time series forecasting models.

## Features to Create
1. Lag features (1, 3, 6, 12 months)
2. Rolling statistics (3, 6, 12 month windows)
3. Time-based features (month, quarter, year)
4. Inflation rates (MoM, YoY)
5. Seasonal indicators

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

In [ ]:
# Load monthly aggregated data
df = pd.read_csv('../data/clean/wfp_monthly_avg.csv', parse_dates=['date'])
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Focus on Maize (White) - Retail for Modeling

In [ ]:
# Filter to Maize retail prices, aggregate to national average
maize = df[df['commodity'] == 'Maize (white) - Retail'].copy()
maize_national = maize.groupby('date').agg({
    'price_mean': 'mean',
    'obs_count': 'sum'
}).reset_index()
maize_national.columns = ['date', 'price', 'observations']
maize_national = maize_national.sort_values('date').reset_index(drop=True)

print(f"Maize national dataset: {maize_national.shape}")
print(f"Date range: {maize_national['date'].min()} to {maize_national['date'].max()}")
maize_national.head(10)

## 2. Create Time-Based Features

In [ ]:
# Extract time components
maize_national['year'] = maize_national['date'].dt.year
maize_national['month'] = maize_national['date'].dt.month
maize_national['quarter'] = maize_national['date'].dt.quarter
maize_national['day_of_year'] = maize_national['date'].dt.dayofyear

# Cyclical encoding for month
maize_national['month_sin'] = np.sin(2 * np.pi * maize_national['month'] / 12)
maize_national['month_cos'] = np.cos(2 * np.pi * maize_national['month'] / 12)

print("✓ Time-based features created")
maize_national[['date', 'price', 'year', 'month', 'quarter', 'month_sin', 'month_cos']].head()

## 3. Create Lag Features

In [ ]:
# Create lag features
for lag in [1, 3, 6, 12]:
    maize_national[f'price_lag_{lag}m'] = maize_national['price'].shift(lag)

print("✓ Lag features created")
cols = ['date', 'price'] + [f'price_lag_{i}m' for i in [1, 3, 6, 12]]
maize_national[cols].head(15)

## 4. Create Rolling Statistics

In [ ]:
# Rolling mean and std for different windows
for window in [3, 6, 12]:
    maize_national[f'price_rolling_mean_{window}m'] = maize_national['price'].rolling(window=window).mean()
    maize_national[f'price_rolling_std_{window}m'] = maize_national['price'].rolling(window=window).std()

print("✓ Rolling statistics created")
cols = ['date', 'price', 'price_rolling_mean_3m', 'price_rolling_std_3m', 'price_rolling_mean_12m']
maize_national[cols].head(15)

## 5. Create Inflation Rates

In [ ]:
# Month-over-Month (MoM) inflation
maize_national['inflation_mom'] = ((maize_national['price'] - maize_national['price_lag_1m']) / maize_national['price_lag_1m']) * 100

# Year-over-Year (YoY) inflation
maize_national['inflation_yoy'] = ((maize_national['price'] - maize_national['price_lag_12m']) / maize_national['price_lag_12m']) * 100

print("✓ Inflation rates calculated")
maize_national[['date', 'price', 'inflation_mom', 'inflation_yoy']].tail(20)

## 6. Create Seasonal Indicators

In [ ]:
# Kenya's maize seasons:
# Harvest seasons: April-May (long rains), Oct-Nov (short rains)
# Lean season: Jan-Mar

def get_season(month):
    if month in [4, 5]:
        return 'harvest_long'
    elif month in [10, 11]:
        return 'harvest_short'
    elif month in [1, 2, 3]:
        return 'lean'
    else:
        return 'normal'

maize_national['season'] = maize_national['month'].apply(get_season)

# One-hot encode seasons
season_dummies = pd.get_dummies(maize_national['season'], prefix='season')
maize_national = pd.concat([maize_national, season_dummies], axis=1)

print("✓ Seasonal indicators created")
print(maize_national['season'].value_counts())

## 7. Save Engineered Dataset

In [ ]:
# Save full feature-engineered dataset
maize_national.to_csv('../data/clean/maize_features.csv', index=False)
print(f"✓ Saved: maize_features.csv ({len(maize_national)} rows, {len(maize_national.columns)} columns)")

# Also save a version without NaN (for ML models)
maize_ml = maize_national.dropna()
maize_ml.to_csv('../data/clean/maize_features_ml.csv', index=False)
print(f"✓ Saved: maize_features_ml.csv ({len(maize_ml)} rows, {len(maize_ml.columns)} columns)")

print("\n=== Feature Engineering Summary ===")
print(f"Total features: {len(maize_national.columns)}")
print(f"Rows with complete data: {len(maize_ml)}")
print(f"Date range (ML): {maize_ml['date'].min()} to {maize_ml['date'].max()}")

print("\n✓ Feature engineering completed!")